In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anishapanja/tcga-brca-e2-clini-xlsx/TCGA-BRCA-E2-CLINI.xlsx
/kaggle/input/datasets/anishapanja/tcga-brca-a2-clini-xlsx/TCGA-BRCA-A2-CLINI.xlsx


In [2]:
import shutil
import os

# Wipe everything in /kaggle/working clean before starting, 
# so we always begin from a known empty state — this also 
# means the Output panel will reflect a fresh start once 
# this cell runs, since Output mirrors whatever currently 
# sits in /kaggle/working
working_dir = "/kaggle/working"
for item in os.listdir(working_dir):
    item_path = os.path.join(working_dir, item)
    if item == ".virtual_documents":
        continue  # this is Kaggle's internal notebook metadata, leave it alone
    if os.path.isdir(item_path):
        shutil.rmtree(item_path)
    else:
        os.remove(item_path)

print("Working directory cleared")
print(os.listdir(working_dir))

Working directory cleared
[]


In [3]:
import os
import urllib.request
import zipfile
import random
from collections import defaultdict

random.seed(42)
PATCHES_PER_PATIENT = 200

def download_and_verify(url, zip_path, label):
    # Remove anything leftover at this path before starting fresh
    if os.path.exists(zip_path):
        os.remove(zip_path)
    print(f"[{label}] Downloading...")
    urllib.request.urlretrieve(url, zip_path)
    size_gb = os.path.getsize(zip_path) / (1024**3)
    print(f"[{label}] Download finished. Size: {size_gb:.2f} GB")
    print(f"[{label}] Verifying integrity...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        bad_file = zf.testzip()
        if bad_file is not None:
            # Raising an error here stops the whole script immediately
            # rather than silently continuing with broken data
            raise RuntimeError(f"[{label}] Corrupted file: {bad_file}")
    print(f"[{label}] Integrity check passed")

def extract_selective(zip_path, extract_path, label):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        all_names = zf.namelist()
        jpg_names = [n for n in all_names if n.endswith(".jpg")]
        print(f"[{label}] Total patches available: {len(jpg_names)}")

        patches_by_patient = defaultdict(list)
        for name in jpg_names:
            slide_id = name.split("/")[1]
            patient_id = slide_id[:12]
            patches_by_patient[patient_id].append(name)
        print(f"[{label}] Unique patients: {len(patches_by_patient)}")

        extracted_count = 0
        for patient_id, patch_list in patches_by_patient.items():
            selected = (patch_list if len(patch_list) <= PATCHES_PER_PATIENT
                        else random.sample(patch_list, PATCHES_PER_PATIENT))
            for patch_name in selected:
                zf.extract(patch_name, extract_path)
                extracted_count += 1
        print(f"[{label}] Extracted: {extracted_count}")

# ---------------- A2 ----------------
a2_url = "https://zenodo.org/records/5337009/files/TCGA-BRCA-A2-DEEPMED-TILES.zip?download=1"
a2_zip = "/kaggle/working/A2_full.zip"
a2_extract = "/kaggle/working/tiles/A2/extracted_full"

download_and_verify(a2_url, a2_zip, "A2")
extract_selective(a2_zip, a2_extract, "A2")
os.remove(a2_zip)
print("[A2] zip deleted, disk freed\n")

# ---------------- E2 ----------------
e2_url = "https://zenodo.org/records/5337009/files/TCGA-BRCA-E2-DEEPMED-TILES.zip?download=1"
e2_zip = "/kaggle/working/E2_full.zip"
e2_extract = "/kaggle/working/tiles/E2/extracted_full"

download_and_verify(e2_url, e2_zip, "E2")
extract_selective(e2_zip, e2_extract, "E2")
os.remove(e2_zip)
print("[E2] zip deleted, disk freed\n")

print("Both A2 and E2 fully downloaded and extracted.")
os.system("df -h /kaggle/working")

[A2] Downloading...
[A2] Download finished. Size: 11.90 GB
[A2] Verifying integrity...
[A2] Integrity check passed
[A2] Total patches available: 268938
[A2] Unique patients: 100
[A2] Extracted: 20000
[A2] zip deleted, disk freed

[E2] Downloading...
[E2] Download finished. Size: 10.31 GB
[E2] Verifying integrity...
[E2] Integrity check passed
[E2] Total patches available: 214210
[E2] Unique patients: 90
[E2] Extracted: 18000
[E2] zip deleted, disk freed

Both A2 and E2 fully downloaded and extracted.
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.9G   18G  10% /kaggle/working


0

In [4]:
import pandas as pd

def build_patches_table(extracted_root, institution_label):
    blocks_folder = os.path.join(extracted_root, "BLOCKS_NORM_MACENKO")
    slide_folders = os.listdir(blocks_folder)
    records = []
    for slide_folder in slide_folders:
        patient_id = slide_folder[:12]
        slide_path = os.path.join(blocks_folder, slide_folder)
        for patch_file in os.listdir(slide_path):
            if patch_file.endswith(".jpg"):
                records.append({
                    "patient_id": patient_id,
                    "patch_path": os.path.join(slide_path, patch_file),
                    "institution": institution_label
                })
    return pd.DataFrame(records)

a2_patches = build_patches_table("/kaggle/working/tiles/A2/extracted_full", "A2")
e2_patches = build_patches_table("/kaggle/working/tiles/E2/extracted_full", "E2")
all_patches = pd.concat([a2_patches, e2_patches], ignore_index=True)

def clean_clini(path):
    df = pd.read_excel(path)
    df["subtype_clean"] = df["TCGA Subtype"].str.replace("BRCA.", "", regex=False)
    df = df[df["subtype_clean"] != "Normal"]
    return df[["PATIENT", "subtype_clean", "TIL Regional Fraction"]]

clini_a2 = clean_clini("/kaggle/input/datasets/anishapanja/tcga-brca-a2-clini-xlsx/TCGA-BRCA-A2-CLINI.xlsx")
clini_e2 = clean_clini("/kaggle/input/datasets/anishapanja/tcga-brca-e2-clini-xlsx/TCGA-BRCA-E2-CLINI.xlsx")
clini_combined = pd.concat([clini_a2, clini_e2], ignore_index=True)

final = all_patches.merge(clini_combined, left_on="patient_id", right_on="PATIENT", how="inner")
final = final[["patient_id", "institution", "subtype_clean", "TIL Regional Fraction", "patch_path"]]

final.to_csv("/kaggle/working/patches_labeled_combined.csv", index=False)
print("Final combined dataset:", len(final), "patches across", final["patient_id"].nunique(), "patients")
print(final["subtype_clean"].value_counts())
print(final.groupby("institution")["patient_id"].nunique())

Final combined dataset: 34583 patches across 171 patients
subtype_clean
LumA     18386
Basal     7597
LumB      6200
Her2      2400
Name: count, dtype: int64
institution
A2    93
E2    78
Name: patient_id, dtype: int64
